In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

c:\Users\2020d\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
train_df = pd.read_csv("train_en_social_media_addiction.csv")
test_df = pd.read_csv("test_en_social_media_addiction.csv")

In [3]:
train_df.describe()

,age,avg_daily_usage_hours,mental_health_score,social_media_conflicts
count,542.000000,542.000000,542.000000,542.000000
mean,20.664207,4.909594,6.232472,2.857934
std,1.372347,1.238843,1.093341,0.932134
min,18.000000,1.500000,4.000000,0.000000
25%,20.000000,4.100000,5.000000,2.000000
50%,21.000000,4.800000,6.000000,3.000000
75%,22.000000,5.700000,7.000000,4.000000
max,24.000000,8.500000,9.000000,5.000000


In [4]:
train_df.dtypes

id                            str
gender                        str
age                         int64
country                       str
education_level               str
social_media_platform         str
avg_daily_usage_hours     float64
academic_impact               str
bedtime                       str
wake_time                     str
mental_health_score         int64
relationship_status           str
social_media_conflicts      int64
addiction                     str
dtype: object

In [5]:
train_df.head()

,id,gender,age,country,education_level,social_media_platform,avg_daily_usage_hours,academic_impact,bedtime,wake_time,mental_health_score,relationship_status,social_media_conflicts,addiction
0,ID_80,P,21,Maldives,S2,Instagram,4.2,No,20:02,03:50,8,berpasangan,2,No
1,ID_52,P,23,Bahamas,S2,LinkedIn,2.8,No,23:25,06:25,8,berpasangan,1,No
2,ID_486,female,21,Denmark,S1,Twitter,4.6,No,20:32,03:50,7,berpasangan,2,No
3,ID_203,perempuan,19,India,S1,WhatsApp,7.2,Yes,20:00,01:36,5,berpasangan,4,Yes
4,ID_317,laki-laki,21,Denmark,S2,Facebook,2.8,No,20:02,05:02,8,berpasangan,2,No


In [6]:
train_df["education_level"].unique()

<ArrowStringArray>
['S2', 'S1', 'SMA']
Length: 3, dtype: str

In [7]:
train_df.isna().sum()

id                        0
gender                    0
age                       0
country                   0
education_level           0
social_media_platform     0
avg_daily_usage_hours     0
academic_impact           0
bedtime                   0
wake_time                 0
mental_health_score       0
relationship_status       0
social_media_conflicts    0
addiction                 0
dtype: int64

gender needs to be fixed \n
bedtime and wake time gotta be changed into int using unusual encoder
drop id
age, avg daily usage hours scaled
country, social media platform, relationship status one hot
education level ordinal

In [8]:
bin_col = ["academic_impact"]
hot_col = ["social_media_platform", "country", "relationship_status"]
scale_col = ["age", "avg_daily_usage_hours", "mental_health_score", "social_media_conflicts"]
ord_col = ["education_level"]
ord_map = {"SMA":0, "S1":1, "S2":2}
ordinal = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
scaler = StandardScaler()
impute1 = SimpleImputer(strategy="most_frequent")
impute2 = SimpleImputer(strategy="median")
impute3 = SimpleImputer(strategy="median")
impute4 = SimpleImputer(strategy="median")
impute5 = SimpleImputer(strategy="constant", fill_value=-1)

In [9]:
labeltrain = train_df["addiction"]
train_df = train_df.drop(["id", "addiction"], axis=1)

In [10]:
x_train, x_val, y_train, y_val = train_test_split(train_df, labeltrain, stratify=labeltrain)

In [11]:
trainbin = x_train[bin_col]
trainbin.replace({"Yes":1, "No":0}, inplace=True)
trainhot = x_train[hot_col]
trainhot = hot.fit_transform(trainhot)
trainord = x_train[ord_col]
trainord = ordinal.fit_transform(trainord)
trainscale = x_train[scale_col]
trainscale = scaler.fit_transform(trainscale)

In [12]:
fixed_gender = x_train["gender"]
fixed_gender.replace({"perempuan":"P", "female":"P", "male":"L", "laki-laki":"L"}, inplace=True)
fixed_gender.unique()
fixed_gender.replace({"P":1, "L":0}, inplace=True)

437    1
469    0
404    0
392    1
236    0
      ..
376    0
45     0
429    0
25     1
264    0
Name: gender, Length: 406, dtype: object

In [13]:
fixed_gender = fixed_gender.to_numpy().reshape(-1,1)

In [14]:
fixed_gender.shape

(406, 1)

In [15]:
x_out = np.hstack((trainbin, trainhot, trainord, trainscale, fixed_gender))
x_out

array([[1, 0.0, 0.0, ..., -1.1509009211721033, 1.262359230953525, 1],
       [1, 0.0, 0.0, ..., -0.22378629022790883, 1.262359230953525, 0],
       [1, 1.0, 0.0, ..., -1.1509009211721033, 1.262359230953525, 0],
       ...,
       [1, 0.0, 0.0, ..., -0.22378629022790883, 0.1672356246135013, 0],
       [1, 0.0, 1.0, ..., -0.22378629022790883, 0.1672356246135013, 1],
       [1, 0.0, 0.0, ..., -0.22378629022790883, 0.1672356246135013, 0]],
      shape=(406, 100), dtype=object)

In [16]:
x_out[2, :]

array([1, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 1.0, 0.0, -1.1889180854668076, 1.2794032949759235,
       -1.1509009211721033, 1.262359230953525, 0], dtype=object)

In [17]:
timetaken = x_train[["bedtime", "wake_time"]]
timetaken

,bedtime,wake_time
437,20:25,01:07
469,22:24,03:18
404,23:22,05:28
392,20:26,02:56
236,21:47,06:05
...,...,...
376,22:03,05:21
45,23:38,06:26
429,20:37,01:31
25,22:22,05:10


In [18]:
def to_min(s):
    # try:
    h,m = s.split(":")
    return int(h)*60+int(m)
    # except:
    #     print(s, type(s))

timetaken["bedtime"] = timetaken["bedtime"].apply(to_min)
timetaken["wake_time"] = timetaken["wake_time"].apply(to_min)
timetaken

,bedtime,wake_time
437,1225,67
469,1344,198
404,1402,328
392,1226,176
236,1307,365
...,...,...
376,1323,321
45,1418,386
429,1237,91
25,1342,310


In [19]:
x_out = np.hstack((x_out, timetaken))

In [20]:
x_out[0,:]

array([1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0,
       0.0, 0.0, 0.0, -0.4652288160522288, 2.1762507467672387,
       -1.1509009211721033, 1.262359230953525, 1, 1225, 67], dtype=object)

In [21]:
valbin = x_val[bin_col]
valbin.replace({"Yes":1, "No":0}, inplace=True)
valhot = x_val[hot_col]
valhot = hot.transform(valhot)
valord = x_val[ord_col]
valord = ordinal.transform(valord)
valscale = x_val[scale_col]
valscale = scaler.transform(valscale)

In [22]:
val_gender = x_val["gender"]
val_gender.replace({"perempuan":"P", "female":"P", "male":"L", "laki-laki":"L"}, inplace=True)
val_gender.unique()
val_gender.replace({"P":1, "L":0}, inplace=True)
val_gender = val_gender.to_numpy().reshape(-1,1)

In [23]:
valtaken = x_val[["bedtime", "wake_time"]]
valtaken["bedtime"] = valtaken["bedtime"].apply(to_min)
valtaken["wake_time"] = valtaken["wake_time"].apply(to_min)
valtaken

,bedtime,wake_time
402,1388,368
389,1398,450
481,1295,239
493,1279,157
281,1336,292
...,...,...
82,1274,284
113,1354,316
452,1359,351
381,1217,173


In [24]:
val_out = np.hstack((valbin, valhot, valord, valscale, val_gender, valtaken))

In [25]:
val_out[0, :]

array([0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0,
       0.0, 0.0, 1.0, 0.9821497227769286, -1.655733819977473,
       0.7033283407162856, -0.9278879817265223, 1, 1388, 368],
      dtype=object)

In [26]:
test_df

,id,gender,age,country,education_level,social_media_platform,avg_daily_usage_hours,academic_impact,bedtime,wake_time,mental_health_score,relationship_status,social_media_conflicts
0,ID_327,P,20,Australia,S1,Instagram,4.1,No,20:26,04:08,7,lajang,2
1,ID_322,male,22,India,S2,WhatsApp,6.6,Yes,20:20,02:02,5,berpasangan,4
2,ID_477,P,20,Australia,S1,Facebook,4.2,No,20:21,03:57,7,berpasangan,2
3,ID_370,perempuan,19,Spain,S1,Instagram,4.2,Yes,21:05,04:47,6,lajang,3
4,ID_267,laki-laki,19,Uzbekistan,SMA,TikTok,5.5,Yes,20:45,02:09,5,hubungan rumit,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,ID_524,female,20,Spain,S1,TikTok,3.7,Yes,22:54,07:06,6,lajang,3
159,ID_288,P,18,Liechtenstein,SMA,Instagram,5.8,Yes,23:00,04:12,5,lajang,4
160,ID_43,perempuan,20,France,S1,Instagram,4.8,No,20:49,03:55,7,berpasangan,2
161,ID_254,perempuan,20,Maldives,S1,Instagram,4.4,No,20:44,04:20,8,berpasangan,2


In [27]:
testbin = test_df[bin_col]
testbin.replace({"Yes":1, "No":0}, inplace=True)
testhot = test_df[hot_col]
testhot = hot.transform(testhot)
testord = test_df[ord_col]
testord = ordinal.transform(testord)
testscale = test_df[scale_col]
testscale = scaler.transform(testscale)

In [28]:
test_gender = test_df["gender"]
test_gender.replace({"perempuan":"P", "female":"P", "male":"L", "laki-laki":"L"}, inplace=True)
test_gender.unique()
test_gender.replace({"P":1, "L":0}, inplace=True)
test_gender = test_gender.to_numpy().reshape(-1,1)

In [29]:
testtaken = test_df[["bedtime", "wake_time"]]
testtaken["bedtime"] = testtaken["bedtime"].apply(to_min)
testtaken["wake_time"] = testtaken["wake_time"].apply(to_min)
testtaken

,bedtime,wake_time
0,1226,248
1,1220,122
2,1221,237
3,1265,287
4,1245,129
...,...,...
158,1374,426
159,1380,252
160,1249,235
161,1244,260


In [30]:
test_out = np.hstack((testbin, testhot, testord, testscale, test_gender, testtaken))

In [31]:
logistic = LogisticRegression(class_weight="balanced", max_iter=1000)
logistic.fit(x_out, y_train)
f1_score(logistic.predict(val_out), y_val, pos_label="Yes")

c:\Users\2020d\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


np.float64(1.0)

In [32]:
hist = HistGradientBoostingClassifier(class_weight="balanced", max_iter=1000)
hist.fit(x_out, y_train)
f1_score(hist.predict(val_out), y_val, pos_label="Yes")

np.float64(1.0)

In [33]:
sv = SVC(class_weight="balanced")
sv.fit(x_out, y_train)
f1_score(sv.predict(val_out), y_val, pos_label="Yes")

np.float64(0.7837837837837838)

In [40]:
preds_test = sv.predict(test_out)

submission = pd.DataFrame({
    'id'   : test_df['id'],
    'addiction': preds_test
})

submission.to_csv('socialaddiction3.csv', index=False)
print('submission.csv tersimpan!')
submission.head(10)

submission.csv tersimpan!


,id,addiction
0,ID_327,No
1,ID_322,Yes
2,ID_477,Yes
3,ID_370,No
4,ID_267,Yes
5,ID_521,No
6,ID_19,No
7,ID_271,Yes
8,ID_511,No
9,ID_314,No
